# Demonstrasi dan Panduan Menjalankan Mini-BERT Bahasa Indonesia dari Nol
**Proyek Kelompok 7, Mata Kuliah Generative AI**

Notebook ini merupakan antarmuka interaktif dan panduan terpadu untuk mengeksplorasi model **Mini-BERT Bahasa Indonesia** yang telah selesai dilatih dari awal (*from scratch*) tanpa bobot pralatih eksternal.

### Pembagian Peran Anggota Kelompok
* **Anggota 1**: Pipeline data dan prapemrosesan korpus Wikipedia bahasa Indonesia, pembersihan teks, pembuatan tokenizer tingkat kata (*word-level*) 8.000 token, partisi split dokumen deterministik (train, validation, test), pemotongan urutan (*sequence chunking* panjang 64 token), serta penyusunan PyTorch Dataset dan Masked Language Model (MLM) Data Collator (`src/data/`).
* **Anggota 2**: Perancangan arsitektur Mini-BERT lengkap 2-layer Transformer Encoder, 4 attention heads, dimensi embedding 128, feed-forward 512, weight tying antara word embeddings dan output head, fungsi pemuatan model dengan verifikasi integritas hash vocabulary, skrip inferensi `predict.py`, serta eksekusi komputasi GPU (`src/models/`).
* **Anggota 3**: Pipeline pelatihan dan evaluasi (`src/training/`), implementasi training loop, learning rate scheduler linear dengan warmup, sistem checkpointing atomic, eksperimen baseline (`LR = 3e-4`) vs eksperimen pembanding (`LR = 1e-3`), protokol penguncian model (*model freezing*), evaluasi split test independen, serta analisis perbandingan kinerja leksikal vs tanda baca.

### Arsitektur Ringkas Model
* **Tipe Arsitektur**: Transformer Encoder (Mini-BERT untuk Masked Language Modeling)
* **Jumlah Lapisan Encoder**: 2 layer
* **Jumlah Attention Heads**: 4 heads
* **Dimensi Representasi (`d_model`)**: 128
* **Dimensi Feed-Forward (`d_ff`)**: 512
* **Panjang Maksimum Urutan (`max_length`)**: 64 token
* **Ukuran Kosakata (*Vocabulary*)**: 8.000 token tingkat kata (*word-level*)
* **Jumlah Parameter Unik**: 1.453.760 parameter (1,45 juta parameter, berkat penerapan *weight tying*)

### Ringkasan Hasil Uji Coba pada Split Test
* **Masked-Token Loss**: 5.4948
* **Top-1 Accuracy**: 17.4430% (4.722 dari 27.071 target)
* **Top-5 Accuracy**: 31.7535% (8.596 dari 27.071 target)
* **Jumlah Urutan Data**: 27.002 chunk pelatihan (train), 2.685 chunk validasi (val), dan 3.426 chunk pengujian (test)

### Tujuan dan Sifat Notebook Ini
Notebook ini berfungsi sebagai **antarmuka demonstrasi dan panduan eksplorasi** yang secara langsung memanggil modul-modul resmi proyek pada direktori `src/`. Seluruh logika pemrosesan token, forward pass arsitektur, dan evaluasi menggunakan kode sumber yang telah teruji, bukan duplikasi kode di dalam notebook. Model bertugas memprediksi token pengganti pada posisi `[MASK]` berdasarkan konteks bidirectional, dengan kualitas prediksi yang mencerminkan hasil pelatihan dari nol pada korpus pilot.


## 1. Petunjuk Penggunaan dan Lingkungan Komputasi

### Langkah Menjalankan Notebook
1. Buka berkas notebook ini melalui VS Code, JupyterLab, atau Jupyter Notebook.
2. Pilih kernel Python yang sesuai dengan environment proyek (misalnya environment tempat PyTorch dan dependensi proyek terpasang).
3. Jalankan sel secara berurutan (*sequential*) dari atas ke bawah.

### Kebijakan Dependensi
* **Dependensi Dasar Proyek**: Tercantum pada berkas `requirements-data.txt` (`datasets`, `pyarrow`, `fsspec`) dan `requirements-pytorch.txt` (`torch`).
* **Dependensi Tambahan Notebook**: `matplotlib` untuk menampilkan visualisasi grafik perbandingan kurva pelatihan, serta `ipykernel` untuk integrasi kernel Jupyter.
* **Kebijakan Instalasi**: Instalasi paket **tidak dijalankan secara otomatis** saat pengguna memilih *Run All*. Perintah instalasi di bawah ini disajikan sebagai petunjuk dan dinonaktifkan secara default agar tidak mengubah environment aktif pengguna secara diam-diam.
* **Portabilitas**: Notebook ini dirancang independen dari path absolut laptop anggota kelompok dan dapat dijalankan pada komputer mana pun yang memiliki salinan repositori ini.


In [ ]:
# Sel Petunjuk Instalasi (Dinonaktifkan Secara Default)
# Jika dependensi belum terpasang pada environment Anda, jalankan perintah berikut melalui terminal:
#
# pip install -r requirements-data.txt
# pip install -r requirements-pytorch.txt
# pip install matplotlib ipykernel


## 2. Pemeriksaan Workspace dan Perangkat Komputasi

Sel berikut mendeteksi lokasi root proyek secara portabel (baik notebook dijalankan dari root direktori proyek maupun dari dalam subfolder `notebooks/`), menambahkan direktori root ke `sys.path`, memvalidasi ketersediaan artefak utama, serta memeriksa perangkat komputasi.

Secara default, notebook menggunakan perangkat **CPU** agar dapat dicoba secara langsung pada laptop atau komputer tanpa kartu grafis khusus. Jika perangkat komputasi CUDA tersedia, Anda dapat memilih perangkat GPU.


In [ ]:
import os
import sys
from pathlib import Path
import torch

# 1. Menentukan direktori root proyek secara portabel
current_dir = Path.cwd().resolve()
if (current_dir / "src").exists():
    PROJECT_ROOT = current_dir
elif (current_dir.parent / "src").exists():
    PROJECT_ROOT = current_dir.parent
else:
    PROJECT_ROOT = current_dir

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Direktori Root Proyek : {PROJECT_ROOT}")

# 2. Validasi keberadaan modul kode dan artefak utama
jalur_penting = {
    "Modul Data (src/data)"                 : PROJECT_ROOT / "src" / "data",
    "Modul Model (src/models)"             : PROJECT_ROOT / "src" / "models",
    "Modul Training (src/training)"         : PROJECT_ROOT / "src" / "training",
    "Artefak Kosakata (vocab.json)"         : PROJECT_ROOT / "data" / "processed" / "pilot" / "vocab.json",
    "Konfigurasi Tokenizer (config.json)"  : PROJECT_ROOT / "data" / "processed" / "pilot" / "tokenizer_config.json",
    "Model Final Terbaik (model.pt)"        : PROJECT_ROOT / "runs" / "lr_1e3_gpu" / "best_model" / "model.pt",
    "Konfigurasi Model Final (config.json)": PROJECT_ROOT / "runs" / "lr_1e3_gpu" / "best_model" / "config.json",
}

berkas_hilang = [nama for nama, path_cek in jalur_penting.items() if not path_cek.exists()]
if berkas_hilang:
    print(f"[PERINGATAN] Komponen berikut belum ditemukan pada repositori: {berkas_hilang}")
else:
    print("[OK] Seluruh modul kode dan berkas artefak utama terverifikasi lengkap.")

# 3. Pemeriksaan runtime Python dan PyTorch
print(f"Versi Python          : {sys.version.split()[0]}")
print(f"Versi PyTorch         : {torch.__version__}")

# Menentukan perangkat komputasi: CPU sebagai default yang aman
cuda_aktif = torch.cuda.is_available()
nama_gpu = torch.cuda.get_device_name(0) if cuda_aktif else "Tidak ada"
print(f"Ketersediaan CUDA     : {cuda_aktif} (Perangkat: {nama_gpu})")

DEFAULT_DEVICE = "cpu"
print(f"Perangkat Aktif Default: {DEFAULT_DEVICE} (Kompatibel untuk demonstrasi tanpa GPU)")


## 3. Ringkasan Data dan Tokenizer

Dataset proyek ini dibangun dari subset 3.000 artikel Wikipedia bahasa Indonesia bersih, dipartisi secara deterministik tanpa kebocoran leksikal (80% Train, 10% Validation, 10% Test), kemudian dipotong menjadi urutan (*chunks*) dengan panjang maksimum 64 token.

Tokenizer yang digunakan adalah `WordLevelTokenizer` dengan batas kosakata 8.000 kata unik (termasuk 5 token khusus).


In [ ]:
import json
from src.data.tokenizer import WordLevelTokenizer, SPECIAL_TOKEN_IDS

# 1. Menampilkan ringkasan statistik dari manifest sequence chunking aktual
path_manifest_chunk = PROJECT_ROOT / "data" / "processed" / "pilot" / "chunking_manifest.json"
if path_manifest_chunk.exists():
    with open(path_manifest_chunk, "r", encoding="utf-8") as f:
        data_chunk = json.load(f)
    print("=== Statistik Partisi dan Sequence Chunking Dataset Pilot ===")
    print(f"{'Split':<12} | {'Dokumen':<10} | {'Total Chunk':<12} | {'Rata-rata Token Isi':<20} | {'Batas Panjang':<15}")
    print("-" * 75)
    for split_kunci, stat in data_chunk.items():
        print(f"{split_kunci.upper():<12} | {stat['total_documents']:<10,d} | {stat['total_chunks_retained']:<12,d} | {stat['avg_content_tokens_per_chunk']:<20.2f} | {stat['max_length']} token")
else:
    print("Berkas chunking_manifest.json tidak ditemukan.")

# 2. Memuat tokenizer resmi proyek
path_tokenizer = PROJECT_ROOT / "data" / "processed" / "pilot"
tokenizer = WordLevelTokenizer.from_pretrained(str(path_tokenizer))
print(f"\nUkuran Kosakata Resmi  : {tokenizer.vocab_size} token")
print(f"Daftar Token Khusus    : {SPECIAL_TOKEN_IDS}")

# 3. Contoh tokenisasi kalimat standar
teks_sampel = "Indonesia adalah negara kepulauan yang memiliki ribuan pulau indah."
token_hasil = tokenizer.tokenize(teks_sampel)
token_ids = tokenizer.convert_tokens_to_ids(token_hasil)

print(f"\n--- Contoh Tokenisasi Normal ---")
print(f"Kalimat Asli    : '{teks_sampel}'")
print(f"Token Hasil     : {token_hasil}")
print(f"Token IDs       : {token_ids}")

# 4. Contoh kalimat memuat kata di luar kosakata (Out-Of-Vocabulary / UNK)
teks_oov = "Albert Einstein meneliti kuantum gravitasi di laboratorium universitas."
token_oov = tokenizer.tokenize(teks_oov)
ids_oov = tokenizer.convert_tokens_to_ids(token_oov)

print(f"\n--- Contoh Tokenisasi dengan Kata OOV (Out-Of-Vocabulary) ---")
print(f"Kalimat Asli    : '{teks_oov}'")
print(f"Token Hasil     : {token_oov}")
print(f"Token IDs       : {ids_oov}")
print(f"Catatan: Token ID 1 melambangkan [UNK] untuk kata yang tidak masuk dalam 8.000 kosakata teratas.")


## 4. Memuat Model Final Terlatih

Sel berikut memuat model final terbaik proyek dari direktori `runs/lr_1e3_gpu/best_model/`.

Model dimuat menggunakan metode resmi `MiniBertForMaskedLM.from_pretrained()`, yang secara otomatis:
1. Memvalidasi kecocokan konfigurasi `config.json` dengan metadata pada `model.pt`.
2. Memverifikasi kecocokan hash SHA-256 kosakata antara konfigurasi model dengan `vocab.json` pada direktori tokenizer.
3. Memastikan *weight tying* antara word embeddings dan output prediction head tetap terikat secara benar.
4. Mengekstrak metadata `checkpoint_status` untuk memastikan bahwa model yang dimuat berstatus `trained`, bukan inisialisasi bobot acak.


In [ ]:
from src.models.bert import MiniBertForMaskedLM

dir_model_final = PROJECT_ROOT / "runs" / "lr_1e3_gpu" / "best_model"
dir_tokenizer = PROJECT_ROOT / "data" / "processed" / "pilot"

# Memuat model final dengan validasi integritas ketat
model = MiniBertForMaskedLM.from_pretrained(
    load_directory=str(dir_model_final),
    filename="model.pt",
    tokenizer_dir=str(dir_tokenizer),
    require_tokenizer=True,
)

# Menyetel model ke mode evaluasi pada perangkat default
model.to(torch.device(DEFAULT_DEVICE))
model.eval()

# Mengambil metadata status dan parameter unik
status_checkpoint = getattr(model, "checkpoint_status", "unknown")
total_parameter_unik = model.num_parameters(only_trainable=True)

print("=== Informasi Model Final Mini-BERT Terpilih ===")
print(f"Lokasi Checkpoint  : {dir_model_final}")
print(f"Status Pelatihan   : {status_checkpoint} (Model telah melalui proses pelatihan penuh)")
print(f"Total Parameter    : {total_parameter_unik:,} parameter unik (dengan weight tying aktif)")
print(f"Lapisan Encoder    : {model.config.num_layers} layer")
print(f"Kepala Perhatian   : {model.config.num_heads} heads")
print(f"Dimensi Tersembunyi: d_model = {model.config.d_model}, d_ff = {model.config.d_ff}")
print(f"Batas Maksimum     : {model.config.max_position_embeddings} token posisi")
print(f"Hash Kosakata      : {model.config.vocab_sha256}")


## 5. Demonstrasi Prediksi Token [MASK]

Fungsi inferensi resmi `predict_mask()` dari modul `src.models.predict` menerima masukan kalimat bahasa Indonesia yang memuat satu atau lebih token `[MASK]`.

### Karakteristik Inferensi Masked Language Model:
1. **Perhitungan Probabilitas**: Probabilitas kandidat dihitung melalui *softmax* atas seluruh 8.000 token kosakata. Nilai yang ditampilkan adalah probabilitas absolut dari distribusi penuh tanpa normalisasi ulang.
2. **Pengecualian Token Khusus**: Seluruh token khusus (`[PAD]`, `[UNK]`, `[CLS]`, `[SEP]`, `[MASK]`) dikecualikan dari pemeringkatan kandidat.
3. **Multi-Mask**: Jika kalimat memuat lebih dari satu `[MASK]`, setiap posisi dievaluasi secara independen dan paralel dari konteks masukan yang sama, bukan merupakan prediksi autoregresif bertahap.
4. **Batas Urutan**: Panjang masukan dibatasi maksimal 64 token (termasuk `[CLS]` dan `[SEP]`). Jika melebihi batas, masukan ditolak demi mencegah pemotongan konteks sepihak.

### Sel Interaktif: Silakan Ubah Kalimat Masukan
Anda dapat mengubah variabel `kalimat_uji`, `nilai_top_k`, atau `pilihan_device` pada sel di bawah ini untuk mencoba kalimat pilihan Anda.


In [ ]:
from src.models.predict import predict_mask

# ==============================================================================
# Sel Interaktif (Silakan Diedit)
# ==============================================================================
kalimat_uji = "Ibu kota Indonesia adalah [MASK] ."
nilai_top_k = 5
pilihan_device = "cpu"  # Pilihan: 'cpu', 'cuda', atau 'auto'

hasil = predict_mask(
    text=kalimat_uji,
    checkpoint_dir=str(dir_model_final),
    tokenizer_dir=str(dir_tokenizer),
    top_k=nilai_top_k,
    device=pilihan_device,
)

print(f"Kalimat Masukan   : {hasil['text_input']}")
print(f"Perangkat         : {hasil['device_used']}")
print(f"Total Token       : {hasil['num_tokens']} token (termasuk [CLS] dan [SEP])")
print(f"Jumlah [MASK]     : {hasil['num_masks']}")
print(f"Status Model      : {hasil['checkpoint_status']}")

if hasil['num_unk_tokens'] > 0:
    print(f"Kata di Luar Kosakata ([UNK]): {hasil['num_unk_tokens']} kata -> {hasil['unk_words']}")

for mask_info in hasil['predictions']:
    pos = mask_info['sequence_position']
    print(f"\n--- Kandidat Prediksi untuk [MASK] pada Posisi Urutan ke-{pos} ---")
    print(f"{'Peringkat':<10} | {'Token ID':<10} | {'Kata Terprediksi':<20} | {'Probabilitas':<15}")
    print("-" * 65)
    for c in mask_info['candidates']:
        persen = c['probability'] * 100.0
        print(f"{c['rank']:<10} | {c['token_id']:<10} | {c['token']:<20} | {persen:6.2f}% ({c['probability']:.6f})")


### Pengujian Contoh Tetap yang Representatif
Berikut adalah serangkaian contoh masukan tetap yang mencakup konteks semantik standar, struktur gramatikal, masukan dengan beberapa token `[MASK]`, serta kalimat yang memuat kata di luar kosakata (*Out-Of-Vocabulary*).


In [ ]:
daftar_contoh = [
    ("Konteks Semantik Standar", "Indonesia adalah negara [MASK] di Asia Tenggara ."),
    ("Konteks Tata Bahasa / Hubungan", "Presiden memimpin pemerintahan [MASK] dibantu oleh para menteri ."),
    ("Multi-Mask (Dua Posisi Mask)", "bahasa [MASK] adalah bahasa [MASK] di indonesia ."),
    ("Kalimat Memuat Kata OOV", "Albert Einstein merupakan ilmuwan peraih nobel [MASK] ."),
]

for judul, teks in daftar_contoh:
    print("=" * 75)
    print(f"Kategori : {judul}")
    print(f"Masukan  : {teks}")
    res = predict_mask(
        text=teks,
        checkpoint_dir=str(dir_model_final),
        tokenizer_dir=str(dir_tokenizer),
        top_k=3,
        device="cpu",
    )
    if res['num_unk_tokens'] > 0:
        print(f"Kata OOV ([UNK]) terdeteksi: {res['unk_words']}")
    
    for item in res['predictions']:
        posisi = item['sequence_position']
        ringkasan_kandidat = ", ".join([f"'{c['token']}' ({c['probability']*100:.2f}%)" for c in item['candidates']])
        print(f"Posisi ke-{posisi} -> {ringkasan_kandidat}")
print("=" * 75)


### Penanganan Kasus Batas (*Edge Cases*) dan Pesan Validasi
Fungsi inferensi dilengkapi dengan validasi masukan untuk mencegah galat tersembunyi. Sel berikut mendemonstrasikan penanganan terhadap:
1. Masukan yang tidak memuat token `[MASK]`.
2. Masukan yang melebihi kapasitas maksimum 64 token.
3. Permintaan perangkat CUDA pada lingkungan tanpa GPU.

Blok penanganan galat (*try-except*) memastikan sel demonstrasi dapat dijalankan tuntas saat *Run All*.


In [ ]:
kasus_batas = [
    ("Masukan Tanpa Token [MASK]", "Indonesia adalah negara maritim yang sangat kaya."),
    ("Masukan Melebihi Batas 64 Token", "kata " * 70 + "[MASK] ."),
    ("Permintaan CUDA Saat Tidak Tersedia", "Kalimat uji coba [MASK] ."),
]

for label_kasus, teks_uji in kasus_batas:
    print(f"Pengujian: {label_kasus}")
    device_uji = "cuda" if "CUDA" in label_kasus else "cpu"
    try:
        predict_mask(
            text=teks_uji,
            checkpoint_dir=str(dir_model_final),
            tokenizer_dir=str(dir_tokenizer),
            top_k=3,
            device=device_uji,
        )
    except (ValueError, RuntimeError) as e:
        print(f"  [BERHASIL DITANGANI] Pesan Validasi Sistem: {e}\n")
    except Exception as e:
        print(f"  [GALAT LAIN]: {e}\n")


## 6. Hasil Eksperimen yang Tersimpan

Sel berikut membaca catatan riwayat pelatihan dan hasil evaluasi test yang telah disimpan pada direktori `runs/`. 

> **Catatan Penting**: Sel ini **hanya membaca data hasil eksperimen yang telah dilakukan sebelumnya**, bukan menjalankan proses pelatihan atau evaluasi ulang dataset.


In [ ]:
# Memuat riwayat eksperimen baseline, komparasi, dan hasil test akhir
path_hist_baseline = PROJECT_ROOT / "runs" / "baseline_gpu" / "training_history.json"
path_hist_lr1e3 = PROJECT_ROOT / "runs" / "lr_1e3_gpu" / "training_history.json"
path_eval_test = PROJECT_ROOT / "runs" / "lr_1e3_gpu" / "final_test_evaluation.json"

with open(path_hist_baseline, "r", encoding="utf-8") as f:
    hist_b = json.load(f)

with open(path_hist_lr1e3, "r", encoding="utf-8") as f:
    hist_lr = json.load(f)

with open(path_eval_test, "r", encoding="utf-8") as f:
    eval_t = json.load(f)

print("=== Tabel Perbandingan Validasi per Epoch (5 Epoch Pelatihan) ===")
garis_tabel = f"{'Epoch':<6} | {'Val Loss (3e-4)':<16} | {'Val Loss (1e-3)':<16} | {'Top-1 Acc (3e-4)':<18} | {'Top-1 Acc (1e-3)':<18} | {'Top-5 Acc (1e-3)':<18}"
print(garis_tabel)
print("-" * len(garis_tabel))

for b_data, lr_data in zip(hist_b["history"], hist_lr["history"]):
    ep = b_data["epoch"]
    b_l = f"{b_data['val_loss']:.4f}"
    lr_l = f"{lr_data['val_loss']:.4f}"
    b_t1 = f"{b_data['val_top1_acc']*100:.2f}%"
    lr_t1 = f"{lr_data['val_top1_acc']*100:.2f}%"
    lr_t5 = f"{lr_data['val_top5_acc']*100:.2f}%"
    print(f"{ep:<6} | {b_l:<16} | {lr_l:<16} | {b_t1:<18} | {lr_t1:<18} | {lr_t5:<18}")

# Menampilkan metrik evaluasi test yang tersimpan
metrik = eval_t["metrics"]
print(f"\n=== Hasil Evaluasi Split Test Independen (Model Final: LR 1e-3, Epoch 5) ===")
print(f"Masked-Token Loss  : {metrik['masked_loss']:.4f}")
print(f"Top-1 Accuracy     : {metrik['top1_accuracy']*100:.4f}% ({metrik['top1_correct']:,} dari {metrik['total_targets']:,} target)")
print(f"Top-5 Accuracy     : {metrik['top5_accuracy']*100:.4f}% ({metrik['top5_correct']:,} dari {metrik['total_targets']:,} target)")
print(f"Jumlah Chunk Test  : {metrik['evaluated_chunks']:,} chunk dievaluasi (1 chunk diabaikan karena berisi 100% UNK)")
print(f"Nilai exp(Loss)    : {metrik['exp_masked_token_loss']:.2f}")
print(f"Catatan Penting: Nilai exp(masked loss) BUKAN perplexity teks penuh atau pseudo-perplexity standar, melainkan eksponensial nilai cross-entropy loss pada token-token yang di-masking.")


In [ ]:
# Visualisasi Grafik Perbandingan Kurva Pelatihan
try:
    import matplotlib.pyplot as plt

    daftar_epoch = [item["epoch"] for item in hist_b["history"]]
    loss_b = [item["val_loss"] for item in hist_b["history"]]
    loss_lr = [item["val_loss"] for item in hist_lr["history"]]

    top1_b = [item["val_top1_acc"] * 100.0 for item in hist_b["history"]]
    top1_lr = [item["val_top1_acc"] * 100.0 for item in hist_lr["history"]]

    top5_b = [item["val_top5_acc"] * 100.0 for item in hist_b["history"]]
    top5_lr = [item["val_top5_acc"] * 100.0 for item in hist_lr["history"]]

    fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(14, 5))

    # Grafik 1: Validation Loss
    ax_loss.plot(daftar_epoch, loss_b, marker="o", label="Baseline (LR = 3e-4)", color="#2b5c8f", linewidth=2)
    ax_loss.plot(daftar_epoch, loss_lr, marker="s", label="Pembanding (LR = 1e-3)", color="#d95f02", linewidth=2)
    ax_loss.set_title("Perbandingan Validation Loss per Epoch", fontsize=12, fontweight="bold")
    ax_loss.set_xlabel("Epoch")
    ax_loss.set_ylabel("Cross Entropy Loss")
    ax_loss.grid(True, linestyle="--", alpha=0.6)
    ax_loss.legend()

    # Grafik 2: Akurasi Top-1 dan Top-5
    ax_acc.plot(daftar_epoch, top1_b, marker="o", linestyle="--", label="Top-1 (LR = 3e-4)", color="#2b5c8f")
    ax_acc.plot(daftar_epoch, top1_lr, marker="s", linestyle="-", label="Top-1 (LR = 1e-3)", color="#d95f02")
    ax_acc.plot(daftar_epoch, top5_b, marker="^", linestyle="--", label="Top-5 (LR = 3e-4)", color="#7570b3")
    ax_acc.plot(daftar_epoch, top5_lr, marker="d", linestyle="-", label="Top-5 (LR = 1e-3)", color="#1b9e77")
    ax_acc.set_title("Perbandingan Akurasi Validasi (%)", fontsize=12, fontweight="bold")
    ax_acc.set_xlabel("Epoch")
    ax_acc.set_ylabel("Akurasi (%)")
    ax_acc.grid(True, linestyle="--", alpha=0.6)
    ax_acc.legend()

    plt.tight_layout()
    plt.show()
    print("[OK] Grafik perbandingan kurva pelatihan berhasil ditampilkan.")
except ImportError:
    print("[INFO] Pustaka matplotlib belum terpasang. Tampilan visual dilewati, data telah tersaji pada tabel di atas.")


## 7. Keterbatasan Model dan Analisis Metodologis

Hasil prediksi dan evaluasi model perlu dipahami dalam konteks keterbatasan metodologis berikut:

1. **Kosakata Tingkat Kata (*Word-Level Vocabulary*)**:
   Tokenizer menggunakan 8.000 kata frekuensi tertinggi tanpa pemecahan subword (seperti BPE atau WordPiece). Akibatnya, kata-kata jadian dengan imbuhan kompleks atau nama diri yang tidak masuk dalam 8.000 kosakata teratas akan dipetakan menjadi token `[UNK]`, dengan proporsi kata tak dikenal sekitar 12% hingga 14% pada data pengujian.

2. **Batas Panjang Urutan Pendek (64 Token)**:
   Panjang urutan masukan dibatasi maksimal 64 token demi efisiensi komputasi pelatihan awal. Hal ini membatasi pemahaman konteks wacana panjang yang melintasi beberapa kalimat.

3. **Kontribusi Tanda Baca pada Akurasi**:
   Berdasarkan dekomposisi performa, akurasi Top-1 model pada token tanda baca (seperti koma dan titik) mencapai ~53,88%, sedangkan akurasi pada kata konten leksikal berkisar ~10,72%. Model cenderung memprediksi tanda baca dan kata fungsi frekuensi tinggi karena frekuensi kemunculannya yang dominan dalam korpus pelatihan.

4. **Eksperimen Berbasis Seed Tunggal**:
   Kedua eksperimen pelatihan dijalankan menggunakan seed 42 untuk menjaga komparabilitas langsung di bawah batasan waktu dan komputasi, sehingga deviasi stokastik multi-run belum terukur secara statistik.

5. **Aturan Masking 80/10/10**:
   Dari 15% token target yang dipilih, 80% diganti menjadi `[MASK]`, 10% diganti token acak, dan 10% dipertahankan sebagai token asli. Pada 10% kasus di mana token asli dipertahankan, model dapat mempertahankan token tanpa perubahan representation shift.

6. **Kualitas Prediksi yang Masih Terbatas**:
   Model dilatih dari nol murni pada subset pilot 2.393 dokumen Wikipedia (27.002 chunk pelatihan). Model ini membuktikan fungsionalitas arsitektur dan konvergensi loss dari inisialisasi acak, namun belum memiliki generalisasi semantik mendalam layaknya model bahasa komersial skala miliaran parameter.


## 8. Petunjuk Lanjutan dan Reproduktibilitas

### Menjalankan Pengujian Unit Otomatis (*Unit Tests*)
Seluruh komponen pipeline data, tokenizer, arsitektur model, dan pipeline pelatihan dilengkapi dengan pengujian regresi otomatis (95 unit test lolos verifikasi):

```bash
# Jalankan seluruh pengujian unit dari terminal
python -m unittest discover tests
```

### Menjalankan Pelatihan Eksperimen Baru
Jika Anda ingin melatih model baru dengan konfigurasi berbeda:

```bash
# Contoh menjalankan pelatihan melalui terminal
python -m src.training.run_experiment --config configs/lr_1e3_gpu.json
```

> **Perhatian Penting**: Untuk menjaga integritas data pengujian, jangan menimpa folder `runs/lr_1e3_gpu` atau `runs/baseline_gpu`. Jika ingin melatih ulang, buat berkas konfigurasi baru dan tentukan `output_dir` baru (misalnya `runs/eksperimen_baru/`).

### Menjalankan Evaluasi Test Baru pada Model Kustom
Untuk mengevaluasi checkpoint model baru pada data test:

```bash
python -m src.training.evaluate_test \
  --model-dir runs/eksperimen_baru/best_model \
  --data-dir data/processed/pilot \
  --test-file test_chunks.jsonl \
  --tokenizer-dir data/processed/pilot \
  --batch-size 32 \
  --device cpu \
  --eval-seed 42 \
  --output-file runs/eksperimen_baru/final_test_evaluation.json
```
